# 01 — Dataset exploration (CIC-IDS2017)

This notebook profiles the **real** data with the same functions the training script uses (`src/eda.py`). Run it after placing the CSVs in `data/raw/`. Interpretations go under each figure **after** you have looked at the actual output.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
from src import config
pd.set_option("display.max_columns", 50)

## Load

In [ ]:
from src.data_loading import load_cicids
raw, load_report = load_cicids()
print(raw.shape)
load_report.rows_per_file

## Profile: size, types, missing/infinite values, duplicates, constant columns

In [ ]:
from src.eda import profile_dataset
profile = profile_dataset(raw, load_report)
{k: profile[k] for k in ['n_rows','n_numeric_features','memory_mb_float32','duplicate_rows_including_label','duplicate_rows_features_only','constant_columns','near_constant_columns_top_value_ge_99.9pct']}

In [ ]:
pd.DataFrame({'missing': pd.Series(profile['missing_values']), 'infinite': pd.Series(profile['infinite_values'])}).fillna(0).astype(int)

In [ ]:
pd.Series(profile['class_distribution']).to_frame('flows').assign(pct=lambda d: (100*d.flows/d.flows.sum()).round(4))

In [ ]:
pd.DataFrame(profile['labels_per_file']).fillna(0).astype(int)

**Interpretation (write after viewing):** what is the class imbalance? Which columns contain inf/NaN and why (zero-duration flows)? How many exact duplicates?

## Clean (inf → NaN, targets, de-duplication)

In [ ]:
from src.data_preprocessing import clean_dataset, feature_columns
df, clean_report = clean_dataset(raw)
clean_report['deduplication']['rows_after'], clean_report['deduplication']['exact_duplicates_removed'], clean_report['deduplication']['conflicting_label_rows_removed']

In [ ]:
pd.DataFrame({'before': clean_report['deduplication']['rows_before_by_label'], 'removed_exact': clean_report['deduplication']['exact_duplicates_removed_by_label'], 'after': clean_report['deduplication']['rows_after_by_label']}).fillna(0).astype(int)

## Figures

In [ ]:
from src.eda import make_eda_figures
config.ensure_dirs()
feats = [c for c in feature_columns(df) if df[c].std() > 0][:22]
paths = make_eda_figures(profile, df, clean_report['deduplication'], feats)
for p in paths: display(Markdown(f'### {p}')); display(Image(str(ROOT/p)))

**Interpretation (write after viewing each figure):** what does it suggest, and what does it *not* prove? (e.g. separable distributions on one feature do not mean the feature causes the attack; lab-specific patterns may not generalise.)